# Nexora Motors — Limpieza y Transformación del Dataset
**Autor:** María Isabel Durango Pérez  
**Dataset:** Coches de segunda mano — Milanuncios.com  
**Fuente:** https://zenodo.org/records/4674757

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")
print("✅ Librerías cargadas correctamente")

Pandas version: 3.0.2
NumPy version:  2.4.4
✅ Librerías cargadas correctamente


In [4]:
# Carga del dataset original
df = pd.read_csv('data/coches_milanuncios_09_04_2021.csv')

# Verificación inicial
print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()
print(df.dtypes)

Filas:    500
Columnas: 21

url                     str
titulo                  str
referencia            int64
marca                   str
modelo                  str
ano_vehic           float64
km                  float64
combustible             str
puertas             float64
cv                  float64
transmision             str
ubicacion               str
vendedor                str
precio              float64
particular              str
stats_visto           int64
stats_contactado      int64
stats_compartido      int64
stats_favorito        int64
stats_renovados       int64
descripcion             str
dtype: object


In [5]:
# 1. Eliminar filas sin año de fabricación (solo 2)
df = df.dropna(subset=['ano_vehic'])

# 2. Imputar km con mediana por marca
df['km'] = df.groupby('marca')['km'].transform(
    lambda x: x.fillna(x.median())
)
df['km'] = df['km'].fillna(df['km'].median())

# 3. Imputar puertas con moda (5)
df['puertas'] = df['puertas'].fillna(5.0)

# 4. Imputar cv con mediana por marca
df['cv'] = df.groupby('marca')['cv'].transform(
    lambda x: x.fillna(x.median())
)
df['cv'] = df['cv'].fillna(df['cv'].median())

# 5. Imputar transmision con moda (manual)
df['transmision'] = df['transmision'].fillna('manual')

# Verificación
nulos = df.isnull().sum()
print("Nulos restantes:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ Cero nulos en el dataset")
print(f"\nFilas tras limpieza: {df.shape[0]}")

Nulos restantes:
✅ Cero nulos en el dataset

Filas tras limpieza: 498


In [6]:
ANO_REFERENCIA = 2021

# 1. Antigüedad del vehículo en años
df['antiguedad'] = ANO_REFERENCIA - df['ano_vehic'].astype(int)

# 2. Precio por CV (eficiencia económica)
df['precio_por_cv'] = (df['precio'] * 1000 / df['cv']).round(2)

# 3. Segmento de precio
bins   = [0, 5, 10, 20, 35, 999]
labels = ['Económico (<5K)', 'Accesible (5-10K)',
          'Medio (10-20K)', 'Premium (20-35K)', 'Lujo (>35K)']
df['segmento_precio'] = pd.cut(df['precio'], bins=bins, labels=labels)

# 4. Rango de antigüedad
bins_ant   = [0, 3, 7, 12, 999]
labels_ant = ['Casi nuevo (0-3a)', 'Reciente (4-7a)',
              'Seminuevo (8-12a)', 'Clásico (>12a)']
df['rango_antiguedad'] = pd.cut(
    df['antiguedad'], bins=bins_ant,
    labels=labels_ant, include_lowest=True
)

# 5. Normalizar texto
df['ubicacion']   = df['ubicacion'].str.replace('_', ' ').str.title()
df['combustible'] = df['combustible'].str.capitalize()

# 6. Engagement score
df['engagement_score'] = (
    df['stats_visto']      * 0.5  +
    df['stats_favorito']   * 30   +
    df['stats_contactado'] * 100
).round(0)

# Verificación
print("✅ Variables derivadas creadas correctamente")
print()
print(df[['antiguedad', 'segmento_precio',
          'rango_antiguedad', 'precio_por_cv',
          'engagement_score']].head())

✅ Variables derivadas creadas correctamente

   antiguedad    segmento_precio   rango_antiguedad  precio_por_cv  \
0          17    Económico (<5K)     Clásico (>12a)          26.00   
1          16  Accesible (5-10K)     Clásico (>12a)          40.71   
2           4   Premium (20-35K)    Reciente (4-7a)         174.74   
3           1     Medio (10-20K)  Casi nuevo (0-3a)         131.73   
4           7  Accesible (5-10K)    Reciente (4-7a)          76.67   

   engagement_score  
0            4046.0  
1           11632.0  
2            1393.0  
3             331.0  
4             899.0  


In [7]:
# Selección de columnas finales
cols_finales = [
    'marca', 'modelo', 'ano_vehic', 'antiguedad', 'rango_antiguedad',
    'km', 'combustible', 'puertas', 'cv', 'transmision',
    'ubicacion', 'particular', 'precio', 'segmento_precio',
    'precio_por_cv', 'stats_visto', 'stats_favorito',
    'stats_contactado', 'engagement_score'
]

df_clean = df[cols_finales].copy()

# Guardar dataset limpio
df_clean.to_csv('data/coches_clean.csv', index=False)

# Verificación final
print(f"✅ Dataset limpio guardado en data/coches_clean.csv")
print(f"   Filas:      {df_clean.shape[0]}")
print(f"   Columnas:   {df_clean.shape[1]}")
print(f"   Nulos:      {df_clean.isnull().sum().sum()}")
print(f"   Duplicados: {df_clean.duplicated().sum()}")
print()
print("Columnas finales:")
print(df_clean.columns.tolist())

✅ Dataset limpio guardado en data/coches_clean.csv
   Filas:      498
   Columnas:   19
   Nulos:      0
   Duplicados: 0

Columnas finales:
['marca', 'modelo', 'ano_vehic', 'antiguedad', 'rango_antiguedad', 'km', 'combustible', 'puertas', 'cv', 'transmision', 'ubicacion', 'particular', 'precio', 'segmento_precio', 'precio_por_cv', 'stats_visto', 'stats_favorito', 'stats_contactado', 'engagement_score']
